Voreinstellung

In [ ]:
#r "BoSSSpad.dll"
using System;
using BoSSS.Foundation;
using BoSSS.Application.BoSSSpad;
using BoSSS.Solution;
using BoSSS.Solution.Tecplot;
using BoSSS.Foundation.IO;
using ilPSP.Tracing;
using BoSSS.Solution.Utils;
using ilPSP.LinSolvers;
using BoSSS.Solution.NSECommon;
using ilPSP.Connectors.Matlab;
using ilPSP;
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using BoSSS.Foundation.Grid.RefElements;
using System.Collections.Generic;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.XDG;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.AdvancedSolvers;
using System.Linq;
using BoSSS.Foundation.Grid.Aggregation;
using BoSSS.Platform;
using MPI.Wrappers;
using System.Diagnostics;
using System.IO;
using System.Collections;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Gnuplot;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
Init();


In [ ]:
#r "StokesHelical_Ak.dll"

In [ ]:
using StokesHelical_Ak;

Voreinstellungen

## Projekt Name

In [ ]:
string proj = "Helical_Centrifugal";

In [ ]:
BoSSSshell.WorkflowMgm.Init(proj);

In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Loading job

In [ ]:
BoSSSshell.WorkflowMgm.Sessions

In [ ]:
var Last_Session = BoSSSshell.WorkflowMgm.Sessions.Last();

## Loading Field from Last Time Step

In [ ]:
var Pressure = Last_Session.Timesteps.Last().Fields.Single(f => f.Identification == "Pressure");
var ur = Last_Session.Timesteps.Last().Fields.Single(f => f.Identification == "ur");
var uxi = Last_Session.Timesteps.Last().Fields.Single(f => f.Identification == "uxi");
var ueta = Last_Session.Timesteps.Last().Fields.Single(f => f.Identification == "ueta");

In [ ]:
var Last_Control = (Last_Session.GetControl() as HelicalControl);

## Exact Solution


In [ ]:
double MaxAmp = Last_Control.maxAmpli;
double a = Globals.a;
double b = Globals.b;
Func<double[], double> ur_exact = (X) => 0;
Func<double[], double> pressure_exact = (X) => MaxAmp * MaxAmp * X[0] * X[0] *0.5;
Func<double[], double> ueta_exact = (X) => (X[0]/(Math.Sqrt(a * a * X[0] * X[0] + b * b )))*a*MaxAmp* X[0];
Func<double[], double> uxi_exact = (X) => (X[0]/(Math.Sqrt(a * a * X[0] * X[0] + b * b )))*b*MaxAmp;

## L2-Error

In [ ]:
// Deep Copy
SinglePhaseField exSol_ur = ur.CloneAs() as SinglePhaseField;
SinglePhaseField exSol_ueta = ueta.CloneAs() as SinglePhaseField;
SinglePhaseField exSol_uxi = uxi.CloneAs() as SinglePhaseField;
SinglePhaseField exSol_pressure = Pressure.CloneAs() as SinglePhaseField;
SinglePhaseField pressureError = Pressure.CloneAs() as SinglePhaseField;

In [ ]:
exSol_ur.ProjectField(ur_exact);
exSol_ueta.ProjectField(ueta_exact);
exSol_uxi.ProjectField(uxi_exact);
pressureError.ProjectField(pressure_exact);

In [ ]:
double ur_L2 = ur.L2Error(exSol_ur);
double uxi_L2 = uxi.L2Error(exSol_uxi);
double ueta_L2 = ueta.L2Error(exSol_ueta);
// Substract Mean Value of Pressure
pressureError.Acc(-1, Pressure);
double psiErrorMean = pressureError.GetMeanValueTotal(null);
pressureError.AccConstant(-psiErrorMean);
double pressure_L2 = pressureError.L2Norm();

In [ ]:
new[] {ur_L2, uxi_L2, ueta_L2, pressure_L2 }

## Checking if Errors are below Threshold

In [ ]:
NUnit.Framework.Assert.LessOrEqual(ur_L2, 1.0e-10, $"ur L2 Error out of range: {ur_L2:0.###e-00} (should be close to 0.0)");
NUnit.Framework.Assert.LessOrEqual(uxi_L2, 1.0e-3, $"uxi L2 Error out of range: {uxi_L2:0.###e-00} (should be close to 0.0)");
NUnit.Framework.Assert.LessOrEqual(ueta_L2, 1.0e-3, $"ueta L2 Error out of range: {ueta_L2:0.###e-00} (should be close to 0.0)");
NUnit.Framework.Assert.LessOrEqual(pressure_L2, 1.0e-2, $"pressure L2 Error out of range: {pressure_L2:0.###e-00} (should be close to 0.0)");